# Smartphone Usage Risk Classification — Full Colab Notebook

**Dataset:** `Smartphone_Usage_And_Addiction_Analysis.csv` (10 original columns)

**Important project limitation:** the original 10-column dataset has no measured addiction label. This notebook creates a clearly named *demonstration risk category* using transparent rules. The model therefore learns to imitate these rules. It must **not** be described as clinically validated smartphone-addiction detection. If your project title must remain “Smartphone Addiction Prediction,” explain this limitation honestly to your teacher.

Run cells from top to bottom. Upload the original 10-column CSV when prompted. At the end, the notebook saves `.pkl` model artifacts and downloads them.

## 1. Install/import libraries

In [ ]:
# Most libraries are already available in Google Colab.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42

## 2. Upload the 10-column CSV

In [ ]:
# Select Smartphone_Usage_And_Addiction_Analysis.csv from your computer.
uploaded = files.upload()
csv_file = next(name for name in uploaded.keys() if name.lower().endswith(".csv"))
df = pd.read_csv(csv_file)

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn names:")
print(df.columns.tolist())

## 3. Check and clean the data

In [ ]:
# Remove accidental spaces from column names and duplicate rows.
df.columns = df.columns.str.strip()
df = df.drop_duplicates().copy()

# Convert expected numeric columns to numbers; invalid values become missing.
numeric_cols = [
    "age", "daily_screen_time_hours", "social_media_hours",
    "gaming_hours", "work_study_hours", "sleep_hours",
    "notifications_per_day"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Missing values before model imputation:")
display(df.isnull().sum().to_frame("missing_values"))
print("Shape after duplicate removal:", df.shape)

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Summary statistics for numeric columns.
display(df[numeric_cols].describe().T)

# Distribution of daily screen time.
plt.figure(figsize=(8, 4))
sns.histplot(df["daily_screen_time_hours"].dropna(), bins=25, kde=True)
plt.title("Daily Screen Time Distribution")
plt.xlabel("Daily screen time (hours)")
plt.tight_layout()
plt.show()

# Compare screen time across gender, if gender exists.
if "gender" in df.columns:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=df, x="gender", y="daily_screen_time_hours")
    plt.title("Daily Screen Time by Gender")
    plt.tight_layout()
    plt.show()

# Correlation between numeric variables.
plt.figure(figsize=(9, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Numeric Features")
plt.tight_layout()
plt.show()

## 5. Create a demonstration target label

In [ ]:
# The 10-column CSV does not contain an addiction label.
# This transparent target is for a usage-risk classification demonstration only.
# It is NOT a medical/clinical diagnosis or independently measured addiction label.

def make_demo_target(row):
    social_and_gaming = row["social_media_hours"] + row["gaming_hours"]

    if (row["daily_screen_time_hours"] >= 8) or (
        social_and_gaming >= 5 and row["sleep_hours"] < 6
    ):
        return "Elevated"
    elif (
        row["daily_screen_time_hours"] >= 5
        or social_and_gaming >= 3
        or row["sleep_hours"] < 7
    ):
        return "Moderate"
    else:
        return "Low"

df["usage_risk_category_demo"] = df.apply(make_demo_target, axis=1)

print("Demo target distribution:")
display(df["usage_risk_category_demo"].value_counts().to_frame("count"))
display((df["usage_risk_category_demo"].value_counts(normalize=True) * 100)
        .round(2).to_frame("percentage"))

sns.countplot(data=df, x="usage_risk_category_demo",
              order=["Low", "Moderate", "Elevated"])
plt.title("Rule-Generated Demonstration Categories")
plt.tight_layout()
plt.show()

## 6. Choose features and split the data

In [ ]:
# IDs are not used as predictive features.
feature_cols = [
    "age", "gender", "daily_screen_time_hours", "social_media_hours",
    "gaming_hours", "work_study_hours", "sleep_hours",
    "notifications_per_day"
]
target_col = "usage_risk_category_demo"

X = df[feature_cols].copy()
y = df[target_col].copy()

numeric_features = [
    "age", "daily_screen_time_hours", "social_media_hours",
    "gaming_hours", "work_study_hours", "sleep_hours",
    "notifications_per_day"
]
categorical_features = ["gender"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 7. Preprocess and train multiple classification algorithms

In [ ]:
# Numeric values: fill missing values with median, then scale.
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical values: fill missing values with most common value, then one-hot encode.
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced"
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE, class_weight="balanced", max_depth=8
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE,
        class_weight="balanced", max_depth=12
    )
}

trained_models = {}
results = []

for model_name, classifier in models.items():
    # Pipeline keeps preprocessing and model together, preventing mismatch at prediction time.
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", classifier)
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    trained_models[model_name] = pipeline
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision_macro": precision_score(y_test, predictions, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_test, predictions, average="macro", zero_division=0),
        "F1_macro": f1_score(y_test, predictions, average="macro", zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values(
    by=["F1_macro", "Accuracy"], ascending=False
).reset_index(drop=True)
display(results_df.round(4))

## 8. Evaluate the selected model

In [ ]:
# Select the model using macro F1 first, then accuracy.
best_model_name = results_df.loc[0, "Model"]
best_model = trained_models[best_model_name]
y_pred = best_model.predict(X_test)

print("Selected model:", best_model_name)
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, labels=["Low", "Moderate", "Elevated"], xticks_rotation=20
)
plt.title("Confusion Matrix — Rule-Generated Demo Target")
plt.tight_layout()
plt.show()

print("Reminder: these metrics measure agreement with the rule-generated label,")
print("not real-world addiction detection accuracy.")

## 9. Save all `.pkl` files

In [ ]:
# Save the full preprocessing + classifier pipeline as one file.
with open("smartphone_usage_risk_model.pkl", "wb") as file:
    pickle.dump(best_model, file)

# Save the exact feature order expected by the model.
with open("smartphone_usage_feature_columns.pkl", "wb") as file:
    pickle.dump(feature_cols, file)

# Save possible output categories.
with open("smartphone_usage_target_classes.pkl", "wb") as file:
    pickle.dump(sorted(y.unique().tolist()), file)

# Save metadata documenting the model and its limitation.
metadata = {
    "best_model_name": best_model_name,
    "target_column": target_col,
    "feature_columns": feature_cols,
    "target_is_rule_based_demo": True,
    "target_rules": {
        "Elevated": "daily_screen_time_hours >= 8 OR (social_media_hours + gaming_hours >= 5 AND sleep_hours < 6)",
        "Moderate": "not Elevated AND (daily_screen_time_hours >= 5 OR social_media_hours + gaming_hours >= 3 OR sleep_hours < 7)",
        "Low": "otherwise"
    },
    "warning": "Rule-generated usage-risk category; not a validated addiction diagnosis."
}
with open("smartphone_usage_model_metadata.pkl", "wb") as file:
    pickle.dump(metadata, file)

print("Saved:")
print("- smartphone_usage_risk_model.pkl")
print("- smartphone_usage_feature_columns.pkl")
print("- smartphone_usage_target_classes.pkl")
print("- smartphone_usage_model_metadata.pkl")

## 10. Test a sample prediction

In [ ]:
# Example only. Change these values to test another hypothetical user.
sample_user = pd.DataFrame([{
    "age": 21,
    "gender": "Female",
    "daily_screen_time_hours": 6.5,
    "social_media_hours": 2.5,
    "gaming_hours": 1.0,
    "work_study_hours": 4.0,
    "sleep_hours": 6.5,
    "notifications_per_day": 80
}])[feature_cols]

predicted_category = best_model.predict(sample_user)[0]
print("Predicted demonstration usage-risk category:", predicted_category)
print("This is not a medical diagnosis.")

## 11. Download the model files

In [ ]:
# Download the artifacts individually from Colab.
files.download("smartphone_usage_risk_model.pkl")
files.download("smartphone_usage_feature_columns.pkl")
files.download("smartphone_usage_target_classes.pkl")
files.download("smartphone_usage_model_metadata.pkl")

In [ ]:
# Download all generated PKL files together in one ZIP.
# Run this cell AFTER the model-saving cell above.
import zipfile
from google.colab import files

pkl_files = [
    "smartphone_usage_risk_model.pkl",
    "smartphone_usage_feature_columns.pkl",
    "smartphone_usage_target_classes.pkl",
    "smartphone_usage_model_metadata.pkl"
]

with zipfile.ZipFile("Smartphone_Usage_PKL_Files.zip", "w", zipfile.ZIP_DEFLATED) as zip_file:
    for file_name in pkl_files:
        zip_file.write(file_name)

files.download("Smartphone_Usage_PKL_Files.zip")